In [ ]:
!pip install transformers
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

: 

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")


: 

In [ ]:
train_data.sample(10)

: 

In [1]:
# Random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

NameError: name 'train_data' is not defined

In [ ]:
train_data.shape

#Data pre-processing

import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) #remove lines
    text = re.sub(r"\s+", " ",text) # remove spaces
    text = re.sub(r"<.*?>", " ",text) # remove HTML tags <p> <h>
    text = text.strip().lower()
    return text
    

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

train_data["dialogue"][0]

tokenizer = T5Tokenizer.from_pretrained("t5-small")

# raw data =. tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length = 512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length = 150, truncation=True)

    inputs["labels"] = targets["input_ids"] #token ids  => add to input as labels
    return inputs